# SET UP

In [ ]:
%pip -q install pyspark==4.0.1 google-cloud-storage==3.4.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Hyperparameters Chosen by the user

In [ ]:
# PATH TO DATA

PROJECT_ID = "cs131-vm-project-513902"
BUCKET = "cs131-lab23"
ROOT = "gs://" + BUCKET + "/stock"
DATA_PATH = ROOT + "/combined.parquet"
print(ROOT)
print(DATA_PATH)

gs://cs131-lab23/stock
gs://cs131-lab23/stock/combined.parquet


In [ ]:
STOCK_NAME = "apollohosp" # This name is a parameter chosen by the . The model trained will be trained and predict on the stock chosen by the user.

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

from google.cloud import storage
client = storage.Client(project=PROJECT_ID)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.feature import StringIndexer
from pathlib import Path
from urllib.request import urlretrieve
import torch
import os
import sys


from sklearn.metrics import f1_score, accuracy_score


In [ ]:
CONNECTOR = str(Path("gcs-connector-3.1.10.jar").resolve())
urlretrieve("https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-3.1.10.jar", CONNECTOR)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("CS131-Class23")
    .master("local[2]")
    .config("spark.jars", CONNECTOR)
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "Master:", spark.sparkContext.master)

Spark: 4.0.1 Master: local[2]


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Helper Function

In [ ]:
def is_valid_stock(name):
  valid_stocks = ['adaniports', 'ambujacem', 'apollohosp', 'bajaj-auto', 'atgl', 'adanient', 'adanigreen', 'adanipower', 'asianpaint', 'abb', 'adaniensol', 'axisbank', 'bpcl', 'canbk', 'bajajhfl', 'bhel', 'bajajfinsv', 'bajajhldng', 'bankbaroda', 'bel', 'bhartiartl', 'britannia', 'bajfinance', 'boschltd', 'cipla', 'cgpower', 'coalindia', 'cumminsind', 'dabur', 'dlf', 'drreddy', 'eichermot', 'dmart', 'cholafin', 'divislab', 'enrin', 'godrejcp', 'eternal', 'gail', 'hdfcamc', 'hdfclife', 'heromotoco', 'grasim', 'hal', 'hcltech', 'havells', 'hdfcbank', 'hindzinc', 'icicigi', 'indhotel', 'indigo', 'indusindbk', 'infy', 'hindunilvr', 'hyundai', 'icicibank', 'hindalco', 'icicipruli', 'ioc', 'irctc', 'lodha', 'lt', 'irfc', 'jswenergy', 'jswsteel', 'maruti', 'itc', 'jiofin', 'lici', 'jindalstel', 'kotakbank', 'ltim', 'ltm', 'mm', 'mm_min', 'motherson', 'maxhealth', 'muthootfin', 'naukri', 'nhpc', 'mazdock', 'nestleind', 'nifty 50', 'nifty bank', 'ntpc', 'pfc', 'recltd', 'reliance', 'shreecem', 'shriramfin', 'pidilitind', 'powergrid', 'ongc', 'sbin', 'pnb', 'sbilife', 'tataconsum', 'tatapower', 'techm', 'tmpv', 'solarinds', 'sunpharma', 'tatacap', 'tatasteel', 'tcs', 'torntpharm', 'siemens', 'titan', 'tmcv', 'vedl', 'trent', 'tvsmotor', 'unitdspr', 'vbl', 'ultracemco', 'unionbank', 'wipro', 'zyduslife']
  return name.lower() in valid_stocks

In [ ]:
def filter_by_stock_name(data, stock_name):

  if not is_valid_stock(stock_name):
    raise ValueError("Invalid Stock Name!")
    return
  stock_name = stock_name.upper()
  data.createOrReplaceTempView("Stocks")
  filter_sql = f"""SELECT * FROM Stocks WHERE name = '{stock_name}'"""
  result = spark.sql(filter_sql)
  return result

In [ ]:
def minute_to_daily(data, path=None):
  """
  Coverts Minute by minute by minute stock data to daily data.

  Grouping by day and then finding this:
  Daily Open - Open value at the first minute of the day
  Daily High - Largest value from high column across the entire day
  Daily Low - Smallest value from the low column aross the entire day
  Daily close - Close value at the last minute at the day


  """
  if path:
    data = spark.read.option('header', 'true').option("inferSchema", True).csv(path)
  data = data.withColumnRenamed('date', 'datetime')
  data = data.withColumn('date', F.to_date('datetime'))

  daily_data = data.groupby('date').agg(
      F.min_by("open", 'datetime').alias('open'), # This line takes the open prices from the first minute of day
      F.max('high').alias('high'), # This lines takes the max high across the entire day and sets it as high
      F.min('low').alias('low'), # This lines takes the min low across the entire day and sets it as low
      F.max_by("close", 'datetime').alias("close") # This lines takes the close price from the last minute of the day
  ).orderBy('date')

  #This only has date, open, high, low and close. We need to get Price Change back again.

  daily_data = daily_data.withColumn('prev_close', F.lag('close', 1).over(Window.orderBy('date')))
  # Creates a temp column thats a copy of close, but shifted by 1 day to create price_change
  daily_data = daily_data.withColumn('price_change', F.when(
      F.col('prev_close').isNull(), 'None')
        .when(F.col('close') > F.col('prev_close'), 'Increased')
        .when(F.col('close') < F.col('prev_close'), 'Decreased')
        .otherwise('None')
        )

  daily_data = daily_data.drop('prev_close') # drop prev_close

  return daily_data


In [ ]:
def processing(data):
  window = Window.orderBy("date")
  feature_exprs = []
  for c in ['open', 'high', 'low', 'close']:
      for k in range(90):
          expr = F.col(c) if k == 0 else F.lag(c, k).over(window)
          feature_exprs.append(expr.alias(f"{c}_t_minus_{k}"))

  data = data.select(
      "date",
      F.month("date").alias("month"),
      *feature_exprs,
      F.lead("price_change", 1).over(window).alias("target")
  )

  data = data.filter(
      F.col("open_t_minus_89").isNotNull()
  ).orderBy("date")
  return data

In [ ]:
def train_test_split(data):
  window = Window.orderBy("date")
  count = data.count()
  s_indx = int(count * 0.9)
  indexed = data.withColumn("rn", F.row_number().over(window))

  train_df = indexed.filter(F.col("rn") <= s_indx).drop("rn")
  test_df  = indexed.filter(F.col("rn") >  s_indx).drop("rn")
  indexer = StringIndexer(inputCol="target", outputCol="label", handleInvalid="keep").fit(train_df)
  train_df = indexer.transform(train_df)
  test_df  = indexer.transform(test_df)


  return train_df, test_df




In [ ]:
def create_pipeline():
  features = ["month"] + [f"{c}_t_minus_{k}" for c in ["open", "high", "low", "close"] for k in range(90)]

  assembler = VectorAssembler(
      inputCols=features,
      outputCol='features'
  )

  xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=1,
    device="cuda",
    max_depth=3,
    learning_rate=0.03,
    n_estimators=200,
    min_child_weight=20,
    gamma=1.0,
    reg_alpha=1.0,
    reg_lambda=5.0,
)


  pipeline = Pipeline(stages=[assembler, xgb_classifier])
  return pipeline



In [ ]:
def create_model(train):
  pipeline = create_pipeline()
  return pipeline.fit(train)

In [ ]:
def predict(model, data):
  predictions = model.transform(data)

  next_day = predictions.tail(1)[0]
  prediction = int(next_day.prediction)
  confidence_score = max(next_day.probability).item()

  encoding = {
      0: "Increased",
      1: "Decreased",
      2: 'None'
  }

  print(f"Model predicts tommorow, this stock will {encoding[prediction]} with a Confidence Score of {confidence_score}")

  predictions = predictions.select("prediction").toPandas()

  actual = data.select("label").toPandas()

  return predictions, actual


In [ ]:
def eval(y_true, y_pred):
  acu = accuracy_score(y_true, y_pred)
  print(f"Accuarcy: {(acu*100):.2f}%")

  f1 = f1_score(y_true, y_pred, average='micro')
  print(f"F1 Score: {(f1*100):.2f}%")


# ML pipeline

In [ ]:
data = spark.read.parquet(DATA_PATH) # Reads the Data from the Google Cloud Storage Bucket

data = filter_by_stock_name(data, STOCK_NAME) # Gets only info from the stock specificed

data = minute_to_daily(data) # Converts the Minute by Minute Data into daily data

data = processing(data)  # Converts day by day data into a sliding window of 90 Days per row for ML

train, test = train_test_split(data) # Splits the Data in train and test datasets

In [ ]:
model = create_model(train) # Creates pipeline that turns data into a vector and feeds into XGBoost Model and trains it.


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'multi:softprob', 'device': 'cuda', 'gamma': 1.0, 'learning_rate': 0.03, 'max_depth': 3, 'min_child_weight': 20, 'reg_alpha': 1.0, 'reg_lambda': 5.0, 'num_class': 3, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 200}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


In [ ]:
predictions, actual = predict(model, test)
eval(actual, predictions)

Model predicts tommorow, this stock will Decreased with a Confidence Score of 0.5607008337974548
Accuarcy: 53.73%
F1 Score: 53.73%
